# 🏥 Sistem Chatbot Edukasi Kesehatan Dasar Masyarakat — RAG + LLM
### Penerapan Retrieval-Augmented Generation (RAG) dengan Large Language Model (LLM)
**Judul Tugas:** PENERAPAN RETRIEVAL-AUGMENTED GENERATION (RAG) DENGAN LARGE LANGUAGE MODEL (LLM) UNTUK SISTEM CHATBOT EDUKASI KESEHATAN DASAR MASYARAKAT


---
## ⚙️ TAHAP 1 — PERSIAPAN SISTEM
### 📦 CELL 1 · Install Semua Library


In [1]:
!pip install groq -q
!pip install neo4j -q
!pip install sentence-transformers -q
!pip install rank-bm25 -q
print('✅ Semua library berhasil diinstall!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 6.3 MB/s eta 0:00:00
✅ Semua library berhasil diinstall!


### ⚙️ CELL 2 · Konfigurasi Global


In [ ]:
import os

NEO4J_URI      = ''
NEO4J_USER     = ''
NEO4J_PASSWORD = ''
GROQ_API_KEY   = 'YOUR_GROQ_API_KEY'
LLM_PROVIDER   = 'groq'
LLM_MODELS   = {'groq': 'llama-3.3-70b-versatile'}
ACTIVE_MODEL = LLM_MODELS[LLM_PROVIDER]
EMBED_MODEL  = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

TOP_K         = 5
MAX_CTX_CHARS = 6000
CHUNK_CHARS   = 1200
DOMAIN        = 'Kesehatan Masyarakat'

SOURCE_LABEL = [
    'Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022',
    'Dasar-Dasar Ilmu Kesehatan Masyarakat',
]

TOPIC_SEARCH_TERMS = {
    'Epidemiologi dan Penyakit'  : 'epidemiologi penyakit distribusi penularan transmisi',
    'Promosi Kesehatan'          : 'promosi kesehatan penyuluhan phbs germas edukasi',
    'Gizi dan Kesehatan'         : 'gizi nutrisi stunting asi mpasi gizi seimbang',
    'Kesehatan Lingkungan'       : 'sanitasi air bersih jamban lingkungan polusi',
    'Kesehatan Ibu dan Anak'     : 'ibu hamil kia antenatal persalinan balita imunisasi',
    'Penyakit Menular'           : 'penyakit menular tuberkulosis malaria dbd diare',
    'Penyakit Tidak Menular'     : 'penyakit tidak menular hipertensi diabetes jantung',
    'Pelayanan Kesehatan'        : 'puskesmas pelayanan kesehatan jkn bpjs',
    'Sanitasi dan Hygiene'       : 'sanitasi hygiene cuci tangan kebersihan',
}

os.environ['GROQ_API_KEY']   = GROQ_API_KEY

rag_chat_history = []
rag_session_log  = []

print('✅ Konfigurasi selesai!')
print(f'   Provider : {LLM_PROVIDER.upper()} | Model: {ACTIVE_MODEL}')
print(f'   Domain   : {DOMAIN}')
print(f'   Embed    : {EMBED_MODEL}')
print()
print('📚 Sumber Buku:')
for src in SOURCE_LABEL:
    print(f'   • {src}')


✅ Konfigurasi selesai!
   Provider : GROQ | Model: llama-3.3-70b-versatile
   Domain   : Kesehatan Masyarakat
   Embed    : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

📚 Sumber Buku:
   • Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022
   • Dasar-Dasar Ilmu Kesehatan Masyarakat


### 🔌 CELL 3 · Koneksi Neo4j + LLM Client


In [4]:
from neo4j import GraphDatabase
from groq import Groq
import time

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def test_neo4j():
    with driver.session() as s:
        msg = s.run("RETURN 'Neo4j terhubung!' AS msg").single()['msg']
        print(f'   ✅ {msg}')
test_neo4j()

llm_client = Groq(api_key=GROQ_API_KEY) if LLM_PROVIDER == 'groq' else None

def call_llm(messages, max_tokens=2000, temperature=0.3, retries=3):
    """Panggil LLM dengan retry otomatis jika rate limit."""
    for attempt in range(retries):
        try:
            return llm_client.chat.completions.create(
                model=ACTIVE_MODEL, messages=messages,
                max_tokens=max_tokens, temperature=temperature, top_p=0.9,
            ).choices[0].message.content
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f'   ⚠️  Retry {attempt+1}/{retries} dalam {wait}s — {e}')
                time.sleep(wait)
            else:
                raise e

try:
    reply = call_llm([{'role':'user','content':'Balas: siap!'}], max_tokens=10)
    print(f'   ✅ LLM OK: "{reply.strip()}"')
    print(f'   🤖 {LLM_PROVIDER.upper()} — {ACTIVE_MODEL}')
except Exception as e:
    print(f'   ❌ LLM Error: {e}')

   ✅ Neo4j terhubung!
   ✅ LLM OK: "Selamat datang. Ada yang bisa saya b"
   🤖 GROQ — llama-3.3-70b-versatile


### 🧠 CELL 4 · Load Embedding Model


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

print(f'📥 Loading: {EMBED_MODEL}')
embed_model = SentenceTransformer(EMBED_MODEL)
DIM = embed_model.get_sentence_embedding_dimension()
print(f'✅ Embedding model siap! Dimensi: {DIM}')


📥 Loading: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model siap! Dimensi: 384


/tmp/ipykernel_11014/2636589780.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = embed_model.get_sentence_embedding_dimension()


---
## 🔍 TAHAP 2 — RETRIEVAL ENGINE
### 🔄 CELL 5 · Query Transformation (HyDE + Expansion)


In [6]:
import numpy as np

def hyde_transform(query):
    """Buat dokumen hipotetis dari query menggunakan LLM, lalu embed."""
    prompt = (
        'Anda adalah pakar kesehatan masyarakat Indonesia. '
        'Tulis paragraf singkat (3-4 kalimat) yang menjelaskan topik berikut '
        'dari sudut pandang kesehatan masyarakat berbasis bukti:\n\n'
        f'Topik: {query}\n\nParagraf:'
    )
    hypo_doc = call_llm([{'role':'user','content':prompt}], max_tokens=180, temperature=0.35)
    orig_vec = embed_model.encode(query, normalize_embeddings=True)
    hypo_vec = embed_model.encode(hypo_doc, normalize_embeddings=True)
    ensemble = 0.6 * orig_vec + 0.4 * hypo_vec
    ensemble /= np.linalg.norm(ensemble)
    return hypo_doc, ensemble

def transform_query(query):
    try:
        hypo_doc, ensemble = hyde_transform(query)
        return {'original_query': query, 'hyde_document': hypo_doc,
                'ensemble_embedding': ensemble.tolist()}
    except Exception as e:
        print(f'   ⚠️  HyDE gagal, pakai direct embed: {e}')
        vec = embed_model.encode(query, normalize_embeddings=True)
        return {'original_query': query, 'hyde_document': None,
                'ensemble_embedding': vec.tolist()}

print('✅ Query Transformation (HyDE) siap!')
print('   Teknik: HyDE (Hypothetical Document Embedding)')
print('   Bobot  : 60% original query + 40% hypothetical document')


✅ Query Transformation (HyDE) siap!
   Teknik: HyDE (Hypothetical Document Embedding)
   Bobot  : 60% original query + 40% hypothetical document


### 🗄️ CELL 6 · Dense Retrieval dari Neo4j


In [7]:
import numpy as np

_chunk_cache = None

def load_chunks():
    global _chunk_cache
    if _chunk_cache is not None:
        return _chunk_cache
    print('   📥 Loading chunks dari Neo4j...')
    with driver.session() as s:
        rows = s.run("""
            MATCH (c:Chunk)
            RETURN c.chunk_id AS chunk_id, c.content AS content,
                   c.page AS page, c.chapter AS chapter,
                   c.source AS source, c.embedding AS embedding
        """)
        _chunk_cache = [dict(r) for r in rows]
    print(f'   ✅ {len(_chunk_cache)} chunks di-cache dari Neo4j')
    return _chunk_cache

def cosine_sim(a, b):
    va, vb = np.array(a, dtype=np.float32), np.array(b, dtype=np.float32)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-10))

def dense_retrieve(query_vec, top_k=TOP_K):
    chunks = load_chunks()
    scored = []
    for chunk in chunks:
        if chunk.get('embedding'):
            score = cosine_sim(query_vec, chunk['embedding'])
            scored.append({**chunk, 'dense_score': score})
    scored.sort(key=lambda x: x['dense_score'], reverse=True)
    return scored[:top_k]

def retrieve(query, top_k=TOP_K):
    t       = transform_query(query)
    results = dense_retrieve(t['ensemble_embedding'], top_k=top_k)
    for r in results:
        r['transformed'] = t
    return results

def format_rag_context(chunks, label=''):
    if not chunks:
        return '[TIDAK ADA KONTEKS RELEVAN DITEMUKAN]'
    header = (
        f'=== KONTEKS KNOWLEDGE BASE KESEHATAN MASYARAKAT ===\n'
        f'Topik: {label or "Kesehatan Masyarakat"} | '
        f'Sumber: Buku Kesehatan Masyarakat | Chunks: {len(chunks)}\n'
        + '=' * 52 + '\n\n'
    )
    parts, total = [], 0
    for i, c in enumerate(chunks, 1):
        content = c.get('content','')[:CHUNK_CHARS]
        part = (
            f'[REF {i}] Hal.{c.get("page","-")} | '
            f'Score:{c.get("dense_score",0):.3f} | Bab:{str(c.get("chapter","-"))[:45]}\n'
            f'Sumber: {c.get("source","-")[:60]}\n'
            f'{content}\n' + '─'*50 + '\n\n'
        )
        if total + len(part) > MAX_CTX_CHARS:
            break
        parts.append(part)
        total += len(part)
    return header + ''.join(parts) + '=== END CONTEXT ===\n'

print('✅ Dense Retrieval + Format Context siap!')


✅ Dense Retrieval + Format Context siap!


---
## 🤖 TAHAP 3 — RAG Q&A CHATBOT KESEHATAN
### ❓ CELL 7 · Tanya Jawab Edukasi Kesehatan Masyarakat
> Ketik pertanyaan seputar kesehatan masyarakat → jawaban berdasarkan buku referensi


In [9]:
from datetime import datetime
import time

TOP_K = 3               # batasi jumlah dokumen
MAX_TOKENS = 700       # output lebih hemat
RETRY_DELAY = 5        # detik tunggu saat rate limit

def format_rag_context(chunks):
    ctx = []
    for c in chunks:
        text = c.get("text", "")
        page = c.get("page", "-")

        # potong teks agar tidak terlalu panjang
        if len(text) > 300:
            text = text[:300] + "..."

        ctx.append(f"(Hal. {page}) {text}")

    return "\n\n".join(ctx)

def safe_call_llm(messages, max_tokens=700, temperature=0.3):
    try:
        return call_llm(messages, max_tokens=max_tokens, temperature=temperature)

    except Exception as e:
        if "rate_limit" in str(e) or "429" in str(e):
            print(f"⏳ Rate limit kena, tunggu {RETRY_DELAY} detik...")
            time.sleep(RETRY_DELAY)

            # retry dengan token lebih kecil
            return call_llm(messages, max_tokens=500, temperature=temperature)
        else:
            raise e

print('┌' + '─'*58 + '┐')
print('│  💬 Chatbot Edukasi Kesehatan Dasar Masyarakat          │')
print('│  Sumber: Buku Ilmu Kesehatan Masyarakat                  │')
print('└' + '─'*58 + '┘')
print()

print('Contoh pertanyaan:')
print('  "Apa itu stunting dan bagaimana cara mencegahnya?"')
print('  "Bagaimana cara mencegah demam berdarah?"')
print('  "Apa itu PHBS dan penerapannya?"')
print('  "Jelaskan epidemiologi tuberkulosis"')
print('─'*60)

pertanyaan = input('\n❓ Pertanyaan Anda: ').strip()

if not pertanyaan:
    print('⚠️ Tidak ada pertanyaan. Jalankan ulang.')
else:
    print(f'\n🔍 Mencari referensi untuk: "{pertanyaan}"')

    chunks = retrieve(pertanyaan, top_k=TOP_K)
    rag_ctx = format_rag_context(chunks)

    pages   = sorted(set(str(c.get('page','-')) for c in chunks))
    sources = list(set(str(c.get('source','-')) for c in chunks))

    print(f'   ✅ {len(chunks)} chunk | Hal: {pages}')
    print(f'   📚 Sumber: {sources}')
    print(f'   🤖 Generating jawaban...')

    SYS = (
        "Anda adalah chatbot edukasi kesehatan masyarakat.\n"
        "Gunakan Bahasa Indonesia yang sederhana dan mudah dipahami.\n\n"
        "ATURAN:\n"
        "1. Gunakan HANYA informasi dari konteks.\n"
        "2. Sertakan referensi halaman (Hal. XX).\n"
        "3. Jika tidak cukup, katakan jujur.\n"
        "4. Berikan jawaban singkat, jelas, dan bermanfaat.\n"
    )

    USR = (
        f"PERTANYAAN:\n{pertanyaan}\n\n"
        f"KONTEKS:\n{rag_ctx}\n\n"
        "Jawab dengan format:\n"
        "1. 📖 Penjelasan\n"
        "2. 🦠 Fakta Penting\n"
        "3. 🛡️ Pencegahan\n"
        "4. 🏥 Kapan ke Tenaga Kesehatan\n"
        "5. 💡 Tips\n"
        "6. ⚠️ Catatan\n"
    )

    jawaban = safe_call_llm(
        [
            {'role': 'system', 'content': SYS},
            {'role': 'user', 'content': USR}
        ],
        max_tokens=MAX_TOKENS
    )

    # =========================
    rag_chat_history.append({'role':'user','content':pertanyaan})
    rag_chat_history.append({'role':'assistant','content':jawaban})

    rag_session_log.append({
        'timestamp' : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'query'     : pertanyaan,
        'answer'    : jawaban,
        'pages'     : pages,
        'sources'   : sources,
    })

    print('\n' + '═'*60)
    print('📋 JAWABAN EDUKASI KESEHATAN')
    print(f'❓ {pertanyaan}')
    print('═'*60)
    print(jawaban)
    print('═'*60)
    print(f'📚 Halaman: {pages}')
    print(f'📖 Sumber : {", ".join(sources)}')
    print()

    print('💡 Untuk pertanyaan lanjutan → jalankan ulang program')

┌──────────────────────────────────────────────────────────┐
│  💬 Chatbot Edukasi Kesehatan Dasar Masyarakat          │
│  Sumber: Buku Ilmu Kesehatan Masyarakat                  │
└──────────────────────────────────────────────────────────┘

Contoh pertanyaan:
  "Apa itu stunting dan bagaimana cara mencegahnya?"
  "Bagaimana cara mencegah demam berdarah?"
  "Apa itu PHBS dan penerapannya?"
  "Jelaskan epidemiologi tuberkulosis"
────────────────────────────────────────────────────────────

❓ Pertanyaan Anda: Bagaimana cara mencegah demam berdarah?

🔍 Mencari referensi untuk: "Bagaimana cara mencegah demam berdarah?"
   ✅ 3 chunk | Hal: ['254', '255', '261']
   📚 Sumber: ['Buku Digital Ilmu Kesehatan Masyarakat - Atik Badiah 2022']
   🤖 Generating jawaban...

════════════════════════════════════════════════════════════
📋 JAWABAN EDUKASI KESEHATAN
❓ Bagaimana cara mencegah demam berdarah?
════════════════════════════════════════════════════════════
1. 📖 Penjelasan: Demam berdarah adalah 

### 💬 CELL 8 · Tanya Jawab Lanjutan (RAG Multi-turn)


In [10]:
from datetime import datetime

if not rag_session_log:
    print('⚠️  Jalankan CELL 7 terlebih dahulu.')
else:
    n_q = len([m for m in rag_chat_history if m['role']=='user'])
    print('┌' + '─'*58 + '┐')
    print(f'│  💬 Tanya Jawab Lanjutan | Riwayat: {n_q} pertanyaan'.ljust(59) + '│')
    print('└' + '─'*58 + '┘')
    print()
    print('Contoh pertanyaan lanjutan:')
    print('  "Bisakah dijelaskan lebih detail tentang itu?"')
    print('  "Apa saja program pemerintah untuk mengatasi hal tersebut?"')
    print('  "Berapa angka kejadian penyakit ini di Indonesia?"')
    print('─'*60)

    lanjutan = input('\n❓ Pertanyaan lanjutan Anda: ').strip()

    if not lanjutan:
        print('⚠️  Tidak ada pertanyaan.')
    else:
        chunks_lanjut = retrieve(lanjutan, top_k=3)
        ctx_lanjut    = format_rag_context(chunks_lanjut)
        pages_lanjut  = sorted(set(str(c.get('page','-')) for c in chunks_lanjut))

        SYS_L = (
            'Anda adalah chatbot edukasi kesehatan masyarakat dalam sesi tanya jawab lanjutan.\n'
            'Jawab pertanyaan lanjutan berdasarkan konteks RAG dan riwayat percakapan.\n'
            'Gunakan Bahasa Indonesia yang mudah dipahami masyarakat awam.\n'
            'Jika informasi tidak ada di konteks, katakan jujur.\n'
            'Selalu sarankan berkonsultasi ke tenaga kesehatan atau puskesmas terdekat.'
        )
        msgs = [{'role':'system','content':SYS_L}]
        msgs.extend(rag_chat_history[-6:])
        msgs.append({'role':'user','content':f'{lanjutan}\n\n{ctx_lanjut}'})

        print(f'🤖 {LLM_PROVIDER.upper()} menjawab...')
        jawaban = call_llm(msgs, max_tokens=1000, temperature=0.35)

        rag_chat_history.extend([
            {'role':'user','content':lanjutan},
            {'role':'assistant','content':jawaban},
        ])
        rag_session_log.append({
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'type'     : 'followup',
            'query'    : lanjutan,
            'answer'   : jawaban,
            'pages'    : pages_lanjut,
        })

        print()
        print('═'*60)
        print(f'❓ {lanjutan}')
        print('═'*60)
        print(jawaban)
        print('═'*60)
        n_total = len([m for m in rag_chat_history if m['role']=='user'])
        print(f'   Total pertanyaan: {n_total} | Jalankan ulang cell ini untuk lanjut.')


┌──────────────────────────────────────────────────────────┐
│  💬 Tanya Jawab Lanjutan | Riwayat: 1 pertanyaan          │
└──────────────────────────────────────────────────────────┘

Contoh pertanyaan lanjutan:
  "Bisakah dijelaskan lebih detail tentang itu?"
  "Apa saja program pemerintah untuk mengatasi hal tersebut?"
  "Berapa angka kejadian penyakit ini di Indonesia?"
────────────────────────────────────────────────────────────

❓ Pertanyaan lanjutan Anda: Bisakah dijelaskan lebih detail tentang itu?
🤖 GROQ menjawab...

════════════════════════════════════════════════════════════
❓ Bisakah dijelaskan lebih detail tentang itu?
════════════════════════════════════════════════════════════
Maaf, saya tidak dapat menjelaskan lebih detail tentang pencegahan demam berdarah karena informasi yang Anda berikan tidak lengkap dan tidak ada konteks yang jelas. Namun, saya dapat memberikan beberapa tips umum untuk mencegah demam berdarah:

1. **Menghindari gigitan nyamuk**: Gunakan obat nyamuk,

---
## 💾 TAHAP 4 — EXPORT & SIMPAN HASIL
### 💾 CELL 9 · Export JSON + Laporan Sesi


In [11]:
import json, shutil, os
from datetime import datetime
from google.colab import files as colab_files

if not rag_session_log:
    print('⚠️  Tidak ada hasil. Jalankan Cell 7 terlebih dahulu.')
else:
    ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
    SAVE_DIR = '/content/drive/MyDrive/kesmas_chatbot_output'
    os.makedirs(SAVE_DIR, exist_ok=True)

    output_json = {
        'generated_at': datetime.now().isoformat(),
        'pipeline_ver': '1.0',
        'domain'      : DOMAIN,
        'provider'    : LLM_PROVIDER,
        'model'       : ACTIVE_MODEL,
        'embed_model' : EMBED_MODEL,
        'sources'     : SOURCE_LABEL,
        'session_log' : rag_session_log,
    }
    local_json = '/content/chatbot_session_results.json'
    drive_json = os.path.join(SAVE_DIR, f'chatbot_results_{ts}.json')
    with open(local_json, 'w', encoding='utf-8') as f:
        json.dump(output_json, f, indent=2, ensure_ascii=False)
    shutil.copy(local_json, drive_json)

    standalone = [r for r in rag_session_log if r.get('type')=='standalone_qa']
    followups  = [r for r in rag_session_log if r.get('type')=='followup']

    lines = [
        '=' * 65,
        'LAPORAN CHATBOT EDUKASI KESEHATAN DASAR MASYARAKAT',
        'Pipeline: RAG + LLM',
        '=' * 65,
        f'Tanggal  : {datetime.now().strftime("%d %B %Y %H:%M")}',
        f'Sistem   : {LLM_PROVIDER.upper()} — {ACTIVE_MODEL}',
        f'Domain   : {DOMAIN}',
        f'Sumber   : {chr(10).join(SOURCE_LABEL)}',
        '',
    ]

    if standalone:
        lines += ['━━━ TANYA JAWAB KESEHATAN ' + '━'*40]
        for i, r in enumerate(standalone, 1):
            lines += [
                f'\n[{i}] ❓ PERTANYAAN: {r.get("query","")}',
                f'    📚 Referensi : Hal. {r.get("pages",[])}',
                f'    📖 Sumber    : {", ".join(r.get("sources",[]))}',
                f'    💬 JAWABAN:',
                '-' * 65,
                r.get('answer',''),
                '',
            ]

    if followups:
        lines += ['', '━━━ TANYA JAWAB LANJUTAN ' + '━'*40]
        for fup in followups:
            lines += [
                f'❓ {fup.get("query","")}',
                f'💬 {fup.get("answer","")}',
                f'📚 Hal. {fup.get("pages",[])}',
                '',
            ]

    lines += [
        '=' * 65,
        '⚠️  CATATAN: Informasi ini bersifat edukatif dan bersumber dari',
        '   buku kesehatan masyarakat. Untuk kondisi kesehatan yang',
        '   memerlukan penanganan, selalu konsultasikan ke tenaga',
        '   kesehatan atau puskesmas terdekat.',
        '=' * 65,
    ]

    local_txt = '/content/laporan_chatbot_kesmas.txt'
    drive_txt = os.path.join(SAVE_DIR, f'laporan_chatbot_{ts}.txt')
    with open(local_txt, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    shutil.copy(local_txt, drive_txt)

    print('═'*62)
    print('💾 HASIL BERHASIL DISIMPAN')
    print('═'*62)
    print(f'   📄 JSON    : {drive_json}')
    print(f'   📋 Laporan : {drive_txt}')
    print()
    print(f'   📊 Ringkasan Sesi:')
    print(f'      • Tanya jawab utama    : {len(standalone)} pertanyaan')
    print(f'      • Tanya jawab lanjutan : {len(followups)} pertanyaan')
    print(f'      • Total interaksi      : {len(rag_session_log)}')
    print()

    colab_files.download(local_json)
    colab_files.download(local_txt)
    print('\n✅ Download JSON + Laporan Sesi berhasil!')

    try: driver.close(); print('\n🔌 Koneksi Neo4j ditutup.')
    except: pass

    print()
    print('═'*62)
    print('🎉 CHATBOT EDUKASI KESEHATAN MASYARAKAT — SELESAI!')
    print('═'*62)
    print('   Pipeline: RAG (Knowledge Graph) + LLM (Generasi Jawaban)')
    print('   Sumber  : Buku Ilmu Kesehatan Masyarakat Indonesia')
    print('   ✅ Cell 1–4 : Persiapan (install, config, koneksi, embed)')
    print('   ✅ Cell 5–6 : Retrieval Engine (HyDE + Dense Cosine)')
    print('   ✅ Cell 7–8 : Chatbot Q&A (single + multi-turn)')
    print('   ✅ Cell 9   : Export JSON + Laporan')


══════════════════════════════════════════════════════════════
💾 HASIL BERHASIL DISIMPAN
══════════════════════════════════════════════════════════════
   📄 JSON    : /content/drive/MyDrive/kesmas_chatbot_output/chatbot_results_20260429_161349.json
   📋 Laporan : /content/drive/MyDrive/kesmas_chatbot_output/laporan_chatbot_20260429_161349.txt

   📊 Ringkasan Sesi:
      • Tanya jawab utama    : 0 pertanyaan
      • Tanya jawab lanjutan : 1 pertanyaan
      • Total interaksi      : 2



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download JSON + Laporan Sesi berhasil!

🔌 Koneksi Neo4j ditutup.

══════════════════════════════════════════════════════════════
🎉 CHATBOT EDUKASI KESEHATAN MASYARAKAT — SELESAI!
══════════════════════════════════════════════════════════════
   Pipeline: RAG (Knowledge Graph) + LLM (Generasi Jawaban)
   Sumber  : Buku Ilmu Kesehatan Masyarakat Indonesia
   ✅ Cell 1–4 : Persiapan (install, config, koneksi, embed)
   ✅ Cell 5–6 : Retrieval Engine (HyDE + Dense Cosine)
   ✅ Cell 7–8 : Chatbot Q&A (single + multi-turn)
   ✅ Cell 9   : Export JSON + Laporan
